In [57]:
import os
import pickle
import time
import numpy as np
import shutil
import cv2 
from deepface import DeepFace
from PIL import Image
import tkinter as tk
from tkinter import filedialog, simpledialog

# --- MASTER CONFIGURATION ---
ROOT_DIR = "./"  # This will now scan every folder in your project directory
OUTPUT_DB_FILE = "master_fest_database.pkl"
ALLOWED_EXTENSIONS = ('.png', '.jpg', '.jpeg')

MODEL_NAME = "ArcFace"
DETECTOR = "mtcnn"

print("Cell 1: Master Scan Config Loaded.")

Cell 1: Master Scan Config Loaded.


In [58]:
def build_master_database(root_directory, output_file):
    face_database = []
    processed_count = 0
    error_count = 0
    
    print(f"🚀 STARTING MASTER INDEXING: {os.path.abspath(root_directory)}")
    start_time = time.time()

    for root, dirs, files in os.walk(root_directory):
        # IMPORTANT: Skip your own output and environment folders
        dirs[:] = [d for d in dirs if d not in ['.venv', 'myenv', 'My_Matches', '.git', '__pycache__']]

        for filename in files:
            if not filename.lower().endswith(ALLOWED_EXTENSIONS):
                continue
                
            img_path = os.path.join(root, filename)
            processed_count += 1
            
            # Progress tracker every 50 images
            if processed_count % 50 == 0:
                elapsed = (time.time() - start_time) / 60
                print(f"📦 Images Processed: {processed_count} | Time Elapsed: {round(elapsed, 1)} mins")

            try:
                # Fast downscaling
                img = cv2.imread(img_path)
                if img is not None:
                    h, w = img.shape[:2]
                    if max(h, w) > 1024:
                        scale = 1024 / max(h, w)
                        img = cv2.resize(img, (int(w * scale), int(h * scale)))
                
                # enforce_detection=False is critical for big datasets
                results = DeepFace.represent(
                    img_path=img, 
                    model_name=MODEL_NAME, 
                    detector_backend=DETECTOR, 
                    enforce_detection=False
                )
                
                for face in results:
                    face_database.append({
                        "file_path": img_path, 
                        "embedding": face["embedding"]
                    })
            except Exception:
                error_count += 1
                continue
                
    # Final save to disk
    with open(output_file, 'wb') as f:
        pickle.dump(face_database, f)

    total_mins = round((time.time() - start_time) / 60, 2)
    print(f"\n✅ --- MASTER INDEX COMPLETE ---")
    print(f"⏱️ Total Time: {total_mins} minutes")
    print(f"📊 Total Photos Scanned: {processed_count}")
    print(f"👤 Total Faces Indexed: {len(face_database)}")
    print(f"📁 Database: {output_file}")

print("Cell 2: Production Engine Ready.")

Cell 2: Production Engine Ready.


In [59]:
build_master_database(ROOT_DIR, OUTPUT_DB_FILE)

🚀 STARTING MASTER INDEXING: d:\Fest_Photo_Search
📦 Images Processed: 50 | Time Elapsed: 2.0 mins
📦 Images Processed: 100 | Time Elapsed: 4.1 mins
📦 Images Processed: 150 | Time Elapsed: 6.3 mins
📦 Images Processed: 200 | Time Elapsed: 8.3 mins
📦 Images Processed: 250 | Time Elapsed: 9.9 mins
📦 Images Processed: 300 | Time Elapsed: 11.7 mins
📦 Images Processed: 350 | Time Elapsed: 13.3 mins
📦 Images Processed: 400 | Time Elapsed: 15.1 mins
📦 Images Processed: 450 | Time Elapsed: 17.1 mins
📦 Images Processed: 500 | Time Elapsed: 18.7 mins
📦 Images Processed: 550 | Time Elapsed: 20.5 mins
📦 Images Processed: 600 | Time Elapsed: 22.9 mins
📦 Images Processed: 650 | Time Elapsed: 25.0 mins
📦 Images Processed: 700 | Time Elapsed: 27.0 mins
📦 Images Processed: 750 | Time Elapsed: 29.0 mins
📦 Images Processed: 800 | Time Elapsed: 30.8 mins
📦 Images Processed: 850 | Time Elapsed: 32.5 mins
📦 Images Processed: 900 | Time Elapsed: 34.2 mins
📦 Images Processed: 950 | Time Elapsed: 35.9 mins
📦 Image

In [60]:
import cv2
import numpy as np
import os
import pickle
import shutil

def get_selfie_embeddings(selfie_paths):
    """Fast-processes target selfies using MTCNN + ArcFace."""
    embeddings = []
    for idx, path in enumerate(selfie_paths):
        print(f"📸 Fast-processing Input Image {idx + 1}...")
        try:
            # DOWNSCALE TARGET SELFIE FOR SPEED
            img = cv2.imread(path)
            if img is not None:
                h, w = img.shape[:2]
                if max(h, w) > 1024:
                    scale = 1024 / max(h, w)
                    img = cv2.resize(img, (int(w * scale), int(h * scale)))
            
            # Use the RAM image directly to skip file I/O bottlenecks
            results = DeepFace.represent(img_path=img, model_name=MODEL_NAME, 
                                         detector_backend=DETECTOR, enforce_detection=True)
            embeddings.append(np.array(results[0]["embedding"]))
            print(f"   -> Success! Feature vector extracted.")
        except Exception as e:
            print(f"   -> ⚠️ Failed to process {os.path.basename(path)}: {e}")
    return embeddings

def find_my_photos(selfie_list, db_file, top_n=30, max_threshold=0.60): 
    print(f"\n📂 Loading Master Database from {db_file}...")
    try:
        with open(db_file, 'rb') as f:
            database = pickle.load(f)
        print(f"   -> Loaded {len(database)} faces from memory.")
    except FileNotFoundError:
        return "❌ Error: Database not found. You must run Cell 3 first."

    # 1. Vectorize your target selfies
    selfie_vectors = get_selfie_embeddings(selfie_list)
    if not selfie_vectors:
        return "❌ Error: Could not generate face profiles from your inputs."

    print("\n⚙️ Comparing your geometry against 10,000+ faces...")
    matches = []
    
    for stored_face in database:
        stored_vector = np.array(stored_face["embedding"])
        best_distance = 1.0 
        
        # Compare against every selfie uploaded to find your 'best' version
        for selfie_vector in selfie_vectors:
            distance = np.dot(selfie_vector, stored_vector) / (np.linalg.norm(selfie_vector) * np.linalg.norm(stored_vector))
            distance = 1 - distance # Cosine distance
            
            if distance < best_distance:
                best_distance = distance
        
        if best_distance <= max_threshold:
            matches.append({
                "path": stored_face["file_path"],
                "distance": best_distance
            })

    # 2. Sort by mathematical proximity (lowest distance = best match)
    matches = sorted(matches, key=lambda x: x["distance"])

    # 3. DEDUPLICATION: Ensure unique file paths in results
    unique_matches = []
    seen_paths = set()
    for match in matches:
        if match["path"] not in seen_paths:
            unique_matches.append(match)
            seen_paths.add(match["path"])

    print(f"\n🏆 --- TOP {min(top_n, len(unique_matches))} UNIQUE MATCHES FOUND ---")
    
    export_folder = "My_Matches"
    os.makedirs(export_folder, exist_ok=True)
    
    # Clean up old results folder
    for f in os.listdir(export_folder):
        file_path = os.path.join(export_folder, f)
        if os.path.isfile(file_path):
            os.remove(file_path)
            
    for i in range(min(top_n, len(unique_matches))):
        match = unique_matches[i]
        folder_name = os.path.basename(os.path.dirname(match["path"]))
        file_name = os.path.basename(match["path"])
        
        # Multiply by 100 to show a clean "Score" instead of a raw decimal
        score = round((1 - match['distance']) * 100, 1)
        print(f"   {i+1}. {folder_name} -> {file_name} (Match Score: {score}%)")
        
        try:
            shutil.copy2(match["path"], os.path.join(export_folder, f"Rank_{i+1}_{file_name}"))
        except Exception as e:
            print(f"      ⚠️ Could not copy {file_name}: {e}")

    print(f"\n✅ SUCCESS: Check the '{export_folder}' folder for your photos!")

print("Cell 4 Complete: High-Speed Search Engine Active.")

Cell 4 Complete: High-Speed Search Engine Active.


In [81]:
import tkinter as tk
from tkinter import filedialog, simpledialog

def select_selfies_and_search():
    root = tk.Tk()
    root.withdraw() 
    root.attributes('-topmost', True) 
    
    print("🖼️ Opening File Explorer... Select 1 to 3 clear target photos.")
    file_paths = filedialog.askopenfilenames(
        title="Select Target Photos",
        filetypes=[("Image files", "*.jpg *.jpeg *.png")]
    )
    
    selected_files = list(file_paths)
    if not selected_files:
        print("⚠️ Search cancelled: No photos selected.")
        root.destroy()
        return
        
    print(f"✅ Target acquired: {len(selected_files)} photo(s).")
    
    # Ask for Number of Matches
    top_n_input = simpledialog.askinteger(
        "Result Count", 
        "How many top matches should I find?", 
        initialvalue=30, minvalue=1, maxvalue=200
    )
    top_n = top_n_input if top_n_input is not None else 30
    
    # Ask for AI Strictness (Threshold)
    threshold_input = simpledialog.askfloat(
        "AI Strictness", 
        "Set AI Threshold (0.40 to 0.70):\n\n0.50 = Very Strict (Only perfect matches)\n0.60 = Balanced (Recommended)\n0.70 = Loose (Finds blurry background shots)", 
        initialvalue=0.60, minvalue=0.1, maxvalue=1.0
    )
    final_threshold = threshold_input if threshold_input is not None else 0.60
    
    root.destroy()
    
    print(f"🚀 Searching Master Database with Strictness: {final_threshold}...")
    
    # TARGETING THE NEW MASTER DATABASE
    find_my_photos(selected_files, "master_fest_database.pkl", top_n=top_n, max_threshold=final_threshold)

# Run the UI!
select_selfies_and_search()

🖼️ Opening File Explorer... Select 1 to 3 clear target photos.
✅ Target acquired: 1 photo(s).
🚀 Searching Master Database with Strictness: 0.6...

📂 Loading Master Database from master_fest_database.pkl...
   -> Loaded 10305 faces from memory.
📸 Fast-processing Input Image 1...
   -> Success! Feature vector extracted.

⚙️ Comparing your geometry against 10,000+ faces...

🏆 --- TOP 10 UNIQUE MATCHES FOUND ---
   1. Desi Sync -> DSC_0785.jpg (Match Score: 100.0%)
   2. Desi Sync -> ARX00833.jpg (Match Score: 53.0%)
   3. Desi Sync -> ARX00626.jpg (Match Score: 48.7%)
   4. Desi Sync -> DSC_0927.jpg (Match Score: 48.5%)
   5. Desi Sync -> DSC03629.jpg (Match Score: 48.2%)
   6. Desi Sync -> DSC_0401.jpg (Match Score: 44.8%)
   7. Hasyamanch -> DSC_0949.jpg (Match Score: 44.7%)
   8. Desi Sync -> DSC_0653.jpg (Match Score: 42.1%)
   9. Desi Sync -> DSC_0505.jpg (Match Score: 41.1%)
   10. Desi Sync -> DSC03726.jpg (Match Score: 40.2%)

✅ SUCCESS: Check the 'My_Matches' folder for your phot